# 11장 동작 분류와 엣지 AI

이 노트북은 웹 교재 [11장 동작 분류와 엣지 AI](https://pidocs.aifusionlab.org/ai-python/ch11/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 실습 자료 내려받기

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/gesture_example.csv
!ls *.csv

## 11.1 가속도 자료의 구간 특징

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch11/11-1/)

In [ ]:
import glob
import pandas as pd

def load_gesture_files(pattern="g_*.csv"):
    """8.5절 파일(g_동작_번호.csv)을 모두 읽어 하나의 DataFrame으로 합친다."""
    parts = []
    for path in glob.glob(pattern):
        _, gesture, student = path.replace(".csv", "").split("_")
        d = pd.read_csv(path)
        d["gesture"], d["student"] = gesture, "S" + student
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("gesture_example.csv")
print(df.shape)
print(df.head(3))
df.groupby(["student", "gesture"]).size().unstack()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for ax, g in zip(axes, ["still", "shake", "tilt", "circle"]):
    d = df[(df["student"] == "S01") & (df["gesture"] == g) & (df["rep"] == 1)]
    for col in ["ax", "ay", "az"]:
        ax.plot(d["t"], d[col], label=col)
    ax.set_title(g)
    ax.set_xlabel("t (s)")
axes[0].set_ylabel("acceleration")
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
W, STEP = 20, 10
FEATS = ["ax_mean", "ay_mean", "az_mean", "ax_std", "ay_std", "az_std", "mag_max"]

def window_features(w):
    """창 하나(DataFrame)의 특징 7개를 딕셔너리로 반환한다."""
    mag = np.sqrt(w["ax"] ** 2 + w["ay"] ** 2 + w["az"] ** 2)
    return {"ax_mean": w["ax"].mean(), "ay_mean": w["ay"].mean(), "az_mean": w["az"].mean(),
            "ax_std": w["ax"].std(), "ay_std": w["ay"].std(), "az_std": w["az"].std(),
            "mag_max": mag.max()}

rows = []
for (student, gesture, rep), d in df.groupby(["student", "gesture", "rep"]):
    d = d.reset_index(drop=True)
    for start in range(0, len(d) - W + 1, STEP):
        f = window_features(d.iloc[start:start + W])
        f.update(student=student, gesture=gesture, rep=rep, start=start)
        rows.append(f)
F = pd.DataFrame(rows)
print(F.shape)
F[["student", "gesture", "rep", "start"] + FEATS].head(3).round(2)

In [ ]:
F.groupby("gesture")[FEATS].mean().round(2)

In [ ]:
plt.figure(figsize=(6, 4.5))
for g in ["still", "shake", "tilt", "circle"]:
    d = F[F["gesture"] == g]
    plt.scatter(d["ax_std"], d["az_mean"], s=10, label=g)
plt.xlabel("ax_std")
plt.ylabel("az_mean")
plt.legend()
plt.show()

## 11.2 kNN과 결정트리 비교

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch11/11-2/)

In [ ]:
train = F[F["rep"] <= 20]
test = F[F["rep"] > 20]
print(len(train), len(test))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn.fit(train[FEATS], train["gesture"])
print("kNN 정확도", round(accuracy_score(test["gesture"], knn.predict(test[FEATS])), 3))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import ConfusionMatrixDisplay

tree = DecisionTreeClassifier(max_depth=3, random_state=0)
tree.fit(train[FEATS], train["gesture"])
p_tree = tree.predict(test[FEATS])
print("결정트리 정확도", round(accuracy_score(test["gesture"], p_tree), 3))
ConfusionMatrixDisplay.from_predictions(test["gesture"], p_tree,
                                        labels=["still", "tilt", "circle", "shake"], cmap="Blues")
plt.show()

## 11.3 결정트리의 조건문 변환

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch11/11-3/)

In [ ]:
from sklearn.tree import export_text

print(export_text(tree, feature_names=FEATS, decimals=4))

In [ ]:
def classify(f):
    """11.2절 결정트리(깊이 3)의 조건문 변환. f는 특징 딕셔너리"""
    if f["ax_std"] <= 0.5996:
        if f["ay_std"] <= 0.4518:
            return "tilt" if f["az_mean"] <= 7.4455 else "still"
        return "tilt" if f["az_mean"] <= 9.5005 else "still"
    if f["ay_std"] <= 0.6770:
        return "circle" if f["ax_std"] <= 5.0755 else "shake"
    return "shake"

pred_if = np.array([classify(r) for r in F[FEATS].to_dict("records")])
pred_tree = tree.predict(F[FEATS])
print("일치 창 수", (pred_if == pred_tree).sum(), "/", len(F))

In [ ]:
def mean(v):
    return sum(v) / len(v)

def std(v):
    m = mean(v)
    return (sum((x - m) ** 2 for x in v) / (len(v) - 1)) ** 0.5

w = df[(df["student"] == "S01") & (df["gesture"] == "shake") & (df["rep"] == 1)].iloc[0:20]
print(round(std(list(w["ax"])), 4), round(w["ax"].std(), 4))
print(round(mean(list(w["az"])), 4), round(w["az"].mean(), 4))

## 11.4 트리 깊이와 정확도의 상충

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch11/11-4/)

In [ ]:
rows = []
for d in [1, 2, 3, 4, 5, 6, 8, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=0).fit(train[FEATS], train["gesture"])
    rows.append({"depth": str(d),
                 "train_acc": round(accuracy_score(train["gesture"], m.predict(train[FEATS])), 3),
                 "test_acc": round(accuracy_score(test["gesture"], m.predict(test[FEATS])), 3),
                 "leaves": m.get_n_leaves(),
                 "rule_lines": len(export_text(m, feature_names=FEATS).splitlines())})
depth_table = pd.DataFrame(rows)
depth_table

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 3.5))
x = range(len(depth_table))
ax1.plot(x, depth_table["test_acc"], marker="o", label="test accuracy")
ax1.plot(x, depth_table["train_acc"], marker="o", label="train accuracy")
ax1.set_xticks(list(x), depth_table["depth"])
ax1.set_xlabel("max_depth")
ax1.set_ylabel("accuracy")
ax2 = ax1.twinx()
ax2.bar(x, depth_table["rule_lines"], alpha=0.2, color="gray", label="rule lines")
ax2.set_ylabel("rule lines")
ax1.legend(loc="center right")
plt.show()